# Figures

All figures use the same style: a fixed colour per cluster number (1 blue, 2 orange, 3 green, 4 yellow,
5 pink) and a blue scale for ordered categories (light = low, dark = high).

Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

### Colours and style

In [ ]:
# one colour per cluster number, and a blue scale for ordered categories
CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
SEQUENTIAL = LinearSegmentedColormap.from_list("blue_ramp", ["#fcfcfb"] + BLUE_RAMP)
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
})

CONTINUOUS = [("Age", "Age (years)"), ("MMSE", "MMSE (0-30)"), ("nWBV", "nWBV"), ("eTIV", "eTIV (cm³)")]
CATEGORICAL = [("CDR", "CDR"), ("Educ", "Education (1-5)"), ("SES", "SES (1-5)"), ("M/F", "Sex")]

### Box plot of one variable by cluster

Box = interquartile range, line = median, diamond = mean.

In [ ]:
def _boxplot(ax, df, labels, var, title):
    clusters = np.sort(np.unique(labels))
    data = [df.loc[labels == c, var].to_numpy() for c in clusters]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55, showmeans=True,
                    medianprops={"color": INK, "linewidth": 1.5},
                    meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": INK, "markersize": 4},
                    whiskerprops={"color": INK_2}, capprops={"color": INK_2},
                    flierprops={"marker": "o", "markersize": 3, "markeredgecolor": INK_2, "alpha": 0.6})
    for patch, c in zip(bp["boxes"], clusters):
        patch.set_facecolor(CLUSTER_COLORS[c - 1])
        patch.set_alpha(0.85)
        patch.set_edgecolor(INK_2)
    ax.set_xticks(range(1, len(clusters) + 1), [str(c) for c in clusters])
    ax.set_title(title)
    ax.set_xlabel("cluster")
    ax.grid(axis="x", visible=False)

### Share of each category within each cluster

In [ ]:
def _stacked(ax, df, labels, var, title):
    ct = pd.crosstab(labels, df[var], normalize="index")
    cats = list(ct.columns)
    colors = (["#9ec5f4", "#256abf"] if var == "M/F"
              else [BLUE_RAMP[int(round(i * (len(BLUE_RAMP) - 1) / max(len(cats) - 1, 1)))] for i in range(len(cats))])
    bottom = np.zeros(len(ct))
    x = np.arange(len(ct))
    for cat, color in zip(cats, colors):
        ax.bar(x, ct[cat], bottom=bottom, color=color, edgecolor="#fcfcfb", linewidth=2, width=0.6,
               label=str(cat).replace(".0", ""))
        bottom += ct[cat].to_numpy()
    ax.set_xticks(x, [str(c) for c in ct.index])
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.5, 1], ["0%", "50%", "100%"])
    ax.set_title(title)
    ax.set_xlabel("cluster")
    ax.grid(axis="x", visible=False)
    ax.legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.0, 1.0), handlelength=1)

### Cluster profile

Age, MMSE, nWBV and eTIV as box plots (top) and CDR, Education, SES and sex as category shares (bottom), by cluster.

In [ ]:
def cluster_profile(df, labels, title):
    """Box plots of the continuous variables and stacked proportions of the categorical ones, by cluster."""
    fig, axes = plt.subplots(2, 4, figsize=(15, 6.4))
    for ax, (var, name) in zip(axes[0], CONTINUOUS):
        _boxplot(ax, df, labels, var, name)
    for ax, (var, name) in zip(axes[1], CATEGORICAL):
        _stacked(ax, df, labels, var, name)
    sizes = ", ".join(f"{c}: n={int((labels == c).sum())}" for c in np.sort(np.unique(labels)))
    fig.suptitle(f"{title}   ({sizes})", fontsize=12, fontweight="bold", color=INK, x=0.01, ha="left")
    fig.tight_layout()
    return fig

### PSI against k

One line per configuration.

In [ ]:
def psi_by_k(psi_aggregate):
    """Aggregate PSI against k, one line per configuration (columns), with values labelled."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for i, name in enumerate(psi_aggregate.columns):
        y = psi_aggregate[name]
        ax.plot(y.index, y.values, color=CLUSTER_COLORS[i], linewidth=2, marker="o", markersize=8,
                markeredgecolor="#fcfcfb", markeredgewidth=2, label=name)
        ax.annotate(name, (y.index[-1], y.values[-1]), xytext=(8, 0), textcoords="offset points",
                    va="center", color=INK, fontsize=9)
    for threshold, label in [(0.147, "small"), (0.33, "medium"), (0.474, "large")]:
        ax.axhline(threshold, color=INK_2, linewidth=0.8, linestyle=(0, (3, 3)))
        ax.text(psi_aggregate.index[0] - 0.15, threshold, label, color=INK_2, fontsize=7, va="bottom", ha="left")
    ax.set_xticks(psi_aggregate.index)
    ax.set_xlabel("number of clusters k")
    ax.set_ylabel("Pairwise Separability Index")
    ax.set_ylim(0, max(0.6, psi_aggregate.to_numpy().max() + 0.05))
    ax.legend(frameon=False, loc="upper right")
    ax.set_title("Aggregate PSI by k")
    fig.tight_layout()
    return fig

### Elbow curve and silhouette by k

In [ ]:
def elbow_and_silhouette(cost, silhouette):
    """Two panels: within-cluster distance relative to k=1 (shape only) and silhouette, one line per configuration.

    cost: DataFrame indexed by k (from 1); silhouette: DataFrame indexed by k (from 2).
    Silhouettes use each configuration's own distance and are not comparable across configurations.
    """
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    rel = cost / cost.iloc[0]
    for i, name in enumerate(cost.columns):
        style = dict(color=CLUSTER_COLORS[i], linewidth=2, marker="o", markersize=8,
                     markeredgecolor="#fcfcfb", markeredgewidth=2, label=name)
        axes[0].plot(rel.index, rel[name], **style)
        axes[1].plot(silhouette.index, silhouette[name], **style)
    axes[0].set_title("Total distance to medoids (relative to k = 1)")
    axes[0].set_ylabel("share of k = 1 total")
    axes[1].set_title("Silhouette width (own distance of each configuration)")
    axes[1].set_ylabel("silhouette")
    axes[1].set_ylim(0, max(0.3, silhouette.to_numpy().max() + 0.05))
    for ax in axes:
        ax.set_xlabel("number of clusters k")
        ax.legend(frameon=False)
    fig.tight_layout()
    return fig

### Stability

Pairwise ARI between the refits, one row per configuration.

In [ ]:
def stability_boxplot(ari_by_config, highlight=None):
    """Horizontal box plots of pairwise ARI between refits, one row per configuration."""
    names = list(ari_by_config)
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(names) + 1.4))
    bp = ax.boxplot([ari_by_config[n] for n in names], orientation="horizontal", patch_artist=True, widths=0.5,
                    showmeans=True, medianprops={"color": INK, "linewidth": 1.5},
                    meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": INK, "markersize": 4},
                    whiskerprops={"color": INK_2}, capprops={"color": INK_2},
                    flierprops={"marker": "o", "markersize": 3, "markeredgecolor": INK_2, "alpha": 0.6})
    for patch, n in zip(bp["boxes"], names):
        patch.set_facecolor(CLUSTER_COLORS[0] if n == highlight else "#9ec5f4")
        patch.set_edgecolor(INK_2)
    ax.set_yticks(range(1, len(names) + 1), [f"{n} (selected)" if n == highlight else n for n in names])
    ax.set_xlim(0, 1)
    ax.set_xlabel("pairwise Adjusted Rand Index between bootstrap refits")
    ax.set_title("Stability of the partitions (30 bootstrap refits)")
    ax.grid(axis="y", visible=False)
    ax.invert_yaxis()
    fig.tight_layout()
    return fig

### PSI before and after

Per-variable PSI of two partitions, side by side.

In [ ]:
def psi_before_after(table, before, after):
    """Dumbbell chart: per-variable PSI of two partitions, one row per variable, with the magnitude thresholds."""
    t = table.sort_values(after)
    y = np.arange(len(t))
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.6))
    for threshold, label in [(0.147, "small"), (0.33, "medium"), (0.474, "large")]:
        ax.axvline(threshold, color=INK_2, linewidth=0.8, linestyle=(0, (3, 3)))
        ax.text(threshold + 0.005, -0.75, label, color=INK_2, fontsize=7, va="center")
    ax.set_ylim(-1, len(t) - 0.5)
    ax.hlines(y, t[before], t[after], color=GRID, linewidth=3, zorder=1)
    ax.scatter(t[before], y, s=70, color=CLUSTER_COLORS[1], edgecolor="#fcfcfb", linewidth=2, zorder=3, label=before)
    ax.scatter(t[after], y, s=70, color=CLUSTER_COLORS[0], edgecolor="#fcfcfb", linewidth=2, zorder=3, label=after)
    for yi, (b, a) in enumerate(zip(t[before], t[after])):
        ax.text(a + (0.025 if a >= b else -0.025), yi, f"{a:.2f}", va="center",
                ha="left" if a >= b else "right", color=INK, fontsize=8)
    ax.set_yticks(y, t.index)
    ax.set_xlim(0, 1)
    ax.set_xlabel("pairwise separability (|Cliff's δ| or TVD), mean over cluster pairs")
    ax.set_title("Separation of each variable between clusters: before vs. after")
    ax.grid(axis="y", visible=False)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    return fig

### Patients as dots

One dot per patient, coloured by cluster.

In [ ]:
def cluster_scatter(ax, x, y, labels, title, xlabel, ylabel):
    """Patients as dots coloured by cluster (fixed slot per cluster number), with a legend giving sizes."""
    for c in np.sort(np.unique(labels)):
        m = labels == c
        ax.scatter(x[m], y[m], s=26, color=CLUSTER_COLORS[c - 1], edgecolor="#fcfcfb", linewidth=0.6,
                   alpha=0.9, label=f"cluster {c} (n={m.sum()})", zorder=2)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False, fontsize=8, loc="best", handletextpad=0.3, markerscale=1.3)

### Scatter-plot matrix of three dimensions

Every pair of the three axes as a scatter plot (below the diagonal) and the distribution of each axis by cluster
(on the diagonal).

In [ ]:
def cluster_scatter_matrix(xyz, labels, title, axis_name="MDS"):
    """3 x 3 scatter-plot matrix: pairwise scatter plots below the diagonal, per-cluster histograms on it."""
    d = xyz.shape[1]
    clusters = np.sort(np.unique(labels))
    fig, axes = plt.subplots(d, d, figsize=(3.1 * d, 3.1 * d))
    for r in range(d):
        for c in range(d):
            ax = axes[r, c]
            if c > r:
                ax.axis("off")
                continue
            if r == c:
                bins = np.histogram_bin_edges(xyz[:, r], bins=20)
                for k in clusters:
                    ax.hist(xyz[labels == k, r], bins=bins, histtype="stepfilled", alpha=0.45,
                            color=CLUSTER_COLORS[k - 1], edgecolor=CLUSTER_COLORS[k - 1], linewidth=1)
                ax.set_yticks([])
            else:
                for k in clusters:
                    m = labels == k
                    ax.scatter(xyz[m, c], xyz[m, r], s=16, color=CLUSTER_COLORS[k - 1], edgecolor="#fcfcfb",
                               linewidth=0.4, alpha=0.9, zorder=2)
                ax.set_yticks([])
            ax.set_xticks([])
            if r == d - 1:
                ax.set_xlabel(f"{axis_name} {c + 1}")
            if c == 0:
                ax.set_ylabel(f"{axis_name} {r + 1}")
    handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=7, color=CLUSTER_COLORS[k - 1],
                          label=f"cluster {k} (n={(labels == k).sum()})") for k in clusters]
    axes[0, d - 1].legend(handles=handles, frameon=False, loc="upper right", fontsize=9)
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK, x=0.01, ha="left")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return fig

### PSI table as a coloured grid

In [ ]:
def psi_heatmap(table, title):
    """Per-variable PSI (rows) by configuration/k (columns), values printed in each cell."""
    fig, ax = plt.subplots(figsize=(1.0 * table.shape[1] + 2.5, 0.45 * table.shape[0] + 1.6))
    im = ax.imshow(table.to_numpy(float), cmap=SEQUENTIAL, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(table.shape[1]), table.columns, rotation=45, ha="right")
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.grid(False)
    for (i, j), v in np.ndenumerate(table.to_numpy(float)):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v > 0.55 else INK)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_title(title)
    fig.tight_layout()
    return fig